# Ensemble Forecasting with IC Perturbations

This notebook walks through how to generate ensemble forecasts from a deterministic
CREDIT model by perturbing the initial conditions.  No special ensemble training is
required — any trained checkpoint works.

**Three perturbation methods are covered:**
1. Gaussian (white) noise — simple, fast baseline
2. Red (colored) noise — spectrally realistic perturbations
3. Bred vectors — dynamically conditioned perturbations using short forecast cycles

**What you need:**
- A trained CREDIT checkpoint
- Your working `model.yml` config (the one you use for deterministic rollout)
- This notebook (or `rollout_metrics_noisy_ic.py` for batch HPC runs)

## 1 — Noise generators

Each noise class takes a state tensor of shape `(batch, channels, time, lat, lon)`
and returns a perturbation tensor of the same shape.  They live in `credit.ensemble`.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from credit.ensemble.gaussian import GaussianNoise
from credit.ensemble.color import ColorNoise
from credit.ensemble.spherical import SphericalNoise
from credit.ensemble.temporal import TemporalNoise

# Dummy state tensor: 1 batch, 4 channels, 1 time step, 32 lat × 64 lon
x = torch.zeros(1, 4, 1, 32, 64)

### 1a — Gaussian (white) noise

Completely uncorrelated random noise at every grid point.  Use as a baseline.

In [ ]:
gaussian = GaussianNoise(amplitude=0.15)
noise_g = gaussian(x)   # shape: (1, 4, 1, 32, 64)

print(f'Shape: {noise_g.shape}')
print(f'Std  : {noise_g.std().item():.3f}  (should be ≈ 0.15)')

plt.figure(figsize=(10, 3))
plt.title('Gaussian noise (channel 0)')
plt.imshow(noise_g[0, 0, 0].numpy(), cmap='RdBu_r', vmin=-0.4, vmax=0.4)
plt.colorbar()
plt.show()

### 1b — Red (colored) noise

`reddening=2` gives brown/red noise (1/f² spectrum) — the default recommended for
atmospheric perturbations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, reddening, label in zip(axes, [0, 1, 2], ['White (0)', 'Pink (1)', 'Brown/Red (2)']):
    noise = ColorNoise(amplitude=0.15, reddening=reddening)(x)
    ax.imshow(noise[0, 0, 0].numpy(), cmap='RdBu_r', vmin=-0.4, vmax=0.4)
    ax.set_title(label)
plt.suptitle('ColorNoise — effect of reddening')
plt.tight_layout()
plt.show()

### 1c — Spherical noise

Matérn covariance on the sphere.  Most physically consistent for lat/lon grids.
Higher `smoothness` → smoother fields.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, sm, label in zip(axes, [1.5, 4.0], ['smoothness=1.5', 'smoothness=4.0']):
    noise = SphericalNoise(amplitude=0.15, smoothness=sm, length_scale=3.0)(x)
    ax.imshow(noise[0, 0, 0].numpy(), cmap='RdBu_r', vmin=-0.4, vmax=0.4)
    ax.set_title(label)
plt.suptitle('SphericalNoise — effect of smoothness')
plt.tight_layout()
plt.show()

## 2 — Connecting noise amplitude to physical units

Amplitudes are specified in **z-score units** (standard deviations of the normalized
state vector).  The actual perturbation magnitude depends on the per-variable standard
deviation used during z-score normalization.

As a rough guide for the CREDIT 6 h WXFormer:

| Variable | Typical σ | amplitude=0.15 → physical perturbation |
|----------|-----------|----------------------------------------|
| Z500 (m²/s²) | ~2000 | ~300 m²/s² |
| T500 (K) | ~5 | ~0.75 K |
| U500 (m/s) | ~12 | ~1.8 m/s |

Start with `amplitude=0.05–0.15` and check that the day-1 ensemble spread looks
physically reasonable (not too small that members collapse, not so large that they
diverge immediately).

## 3 — Bred vectors

Bred vectors use short forecast cycles to grow perturbations in the directions
the model is most sensitive to.  They require a loaded model and a dataset, so
they are demonstrated conceptually here and exercised in the batch scripts.

**Algorithm summary:**
```
For each member:
    x_pert = x_base + noise(x_base)          # initial random perturbation
    For cycle in 1..num_cycles:
        x_base_evolved = model(x_base, n_steps)
        x_pert_evolved = model(x_pert, n_steps)
        delta = x_pert_evolved - x_base_evolved
        delta = rescale(delta, target_amplitude)
        x_pert = x_base + delta              # reset with rescaled bred perturbation
    IC_member = x_forecast_start + delta     # add final bred vector to forecast IC
```

Key config options:

| Option | Default | Effect |
|--------|---------|--------|
| `num_cycles` | 5 | More cycles → perturbations better aligned with model growth directions |
| `bred_time_lag` | 480 h | Hours before forecast start; longer = more growth |
| `integration_steps` | 1 | Model steps per cycle (1 = one 6 h step) |
| `hemispheric_rescale` | true | Amplify perturbations at high latitudes |
| `perturb_channel_idx` | null | Restrict to one channel (null = all channels) |

## 4 — Temporal noise

`TemporalNoise` wraps any spatial noise generator with an AR(1) process, so perturbations
evolve smoothly across forecast steps rather than being freshly drawn at each step.

In [ ]:
base_noise = GaussianNoise(amplitude=0.1)
temporal   = TemporalNoise(
    noise_generator=base_noise,
    temporal_correlation=0.9,   # AR(1) coefficient
    hemispheric_rescale=False,  # True requires a terrain file
)

# Simulate 10 forecast steps
steps = 10
delta = None
perturbations = []

for step in range(1, steps + 1):
    _, delta = temporal(x, previous_perturbation=delta, forecast_step=step)
    perturbations.append(delta[0, 0, 0, 8, 16].item())  # one grid point

plt.figure(figsize=(8, 3))
plt.plot(perturbations, marker='o')
plt.xlabel('Forecast step')
plt.ylabel('Perturbation value')
plt.title('AR(1) temporal noise at one grid point (ρ=0.9)')
plt.grid(True)
plt.show()

## 5 — Config: adding ensemble to your model.yml

Copy your deterministic `model.yml` and add the `ensemble` block.  Below are the three
most common configurations.

In [ ]:
# Run this cell to print ready-to-paste YAML snippets

configs = {
    'Gaussian noise (baseline)': """
predict:
  mode: none
  ensemble_size: 10
  save_forecast: '/glade/derecho/scratch/<you>/CREDIT_runs/ens_gauss/'

  ensemble:
    noise:
      type: "gaussian"
      amplitude: 0.15
    bred_vector:
      enabled: false
    temporal_noise:
      enabled: false
""",
    'Red noise (recommended)': """
predict:
  mode: none
  ensemble_size: 16
  save_forecast: '/glade/derecho/scratch/<you>/CREDIT_runs/ens_red/'

  ensemble:
    noise:
      type: "red"
      amplitude: 0.15
      reddening: 2
    bred_vector:
      enabled: false
    temporal_noise:
      enabled: false
""",
    'Bred vectors with red noise': """
predict:
  mode: none
  ensemble_size: 16
  save_forecast: '/glade/derecho/scratch/<you>/CREDIT_runs/ens_bred/'

  ensemble:
    noise:
      type: "red"
      amplitude: 0.12
      reddening: 2
    bred_vector:
      enabled: true
      amplitude: 0.12
      num_cycles: 5
      integration_steps: 1
      hemispheric_rescale: true
      bred_time_lag: 480
    temporal_noise:
      enabled: false
""",
}

for label, snippet in configs.items():
    print(f'=== {label} ===\n{snippet}')

## 6 — Running the ensemble

### Option A: Compute metrics only (fast, no full NetCDF saved)

```bash
# Local run
python applications/rollout_metrics_noisy_ic.py -c model.yml

# PBS submission on Derecho/Casper
python applications/rollout_metrics_noisy_ic.py -c model.yml -l 1
```

Output: per-init CSV files in `{save_forecast}/metrics/`
- `{datetime}_ensemble.csv` — per-member RMSE/MAE per channel
- `{datetime}_average.csv`  — ensemble-mean RMSE, spread, CRPS

### Option B: Save full ensemble NetCDFs

Add `ensemble_size: N` to the `predict:` block (same config) and run:

```bash
python applications/rollout_to_netcdf.py -c model.yml
```

Output: one NetCDF per lead time with an `ensemble` dimension (100 members × lat × lon).
These are the files expected by `ensemble_wb2_verif.py` for WeatherBench-style CRPS scoring.

## 7 — Evaluating ensemble output

After running `rollout_to_netcdf.py` to generate ensemble NetCDFs:

In [ ]:
# Example: load one ensemble lead file and compute spatial CRPS manually
import xarray as xr
from credit.verification.ensemble import crps_spatial_avg

# Adjust paths as needed
ENSEMBLE_FILE = '/glade/derecho/scratch/schreck/CREDIT_runs/ensemble/scheduler/netcdf_pressure_interp/2022-01-01T00Z/pred_2022-01-01T00Z_120.nc'
TRUTH_FILE    = None   # set to cesm_stage1 zarr path

# Uncomment when running on NCAR systems:
# ds = xr.open_dataset(ENSEMBLE_FILE)
# Z500_ens = ds['Z_PRES'].sel(pressure=500.0).squeeze('time').values.astype(np.float64)  # (100, lat, lon)
# # load truth at same verification time from cesm_stage1...
# Z500_truth = ...
# lats = ds.latitude.values
# w = np.cos(np.deg2rad(lats)); w = w / w.mean()
# crps, spread = crps_spatial_avg(Z500_ens, Z500_truth, w)
# print(f'Z500 CRPS at 120h: {crps:.1f} m²/s²   spread: {spread:.1f} m²/s²')

print('Demo only — set ENSEMBLE_FILE and TRUTH_FILE to real paths to run')

### WeatherBench-style batch evaluation

For a full 2022 evaluation producing NetCDF output in Kyle-format:

```bash
python applications/ensemble_wb2_verif.py \
    --forecast /glade/derecho/scratch/schreck/CREDIT_runs/ensemble/scheduler/netcdf_pressure_interp \
    --out      /glade/work/schreck/CREDIT_verif/ensemble \
    --tag      ensemble
```

This produces `CRPS_006h_240h_ensemble.nc`, `SPREAD_006h_240h_ensemble.nc`, etc.
See `docs/source/PerformanceMetrics.md` for how to compare with HRES and deterministic runs.

## 8 — Choosing perturbation amplitude

A quick calibration: plot spread vs RMSE at day 1 and day 5 for a few amplitude values.
Ideal spread-skill ratio is 1.0 (spread = RMSE).  Well-calibrated ensembles have
spread that tracks RMSE as a function of lead time.

In [ ]:
# Illustrative spread-skill diagram (synthetic data)
lead_days = np.arange(1, 11)
rmse_ref  = 0.3 * lead_days**0.7      # hypothetical deterministic RMSE growth

for amp_label, amp_factor in [('too small (0.3×)', 0.3), ('calibrated (1.0×)', 1.0), ('too large (2.0×)', 2.0)]:
    spread = rmse_ref * amp_factor * (1 + 0.05 * np.random.randn(len(lead_days)))
    plt.plot(lead_days, spread, label=f'spread {amp_label}')

plt.plot(lead_days, rmse_ref, 'k--', lw=2, label='RMSE (truth)')
plt.xlabel('Lead time (days)')
plt.ylabel('Spread / RMSE (z-score units)')
plt.title('Spread-skill diagram — ideal spread tracks RMSE')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()